# Pipeline overview

Runs the whole validator on photos and shows, for every German ID photo rule (Fotomustertafel), whether it passes, warns or fails and why. Use it to see that the pipeline works end to end; `visualize_checks2.ipynb` digs into the internals of single checks.

- A **check** ends as `pass`, `warn`, `fail`, `skipped` (it couldn't judge this photo, e.g. no face) or `error` (it crashed, which is a bug).
- A **rule** takes the worst status of the checks that cover it and ran. `not_checked` means no check covers the rule yet; it is not a pass.
- Most face checks come from **OFIQ** (the BSI's ISO/IEC 29794-5 implementation). Start Jupyter with `OFIQ_CONFIG_DIR` pointing at its `data/` folder; without it this notebook falls back to `../OFIQ-Project/data` if that exists, and otherwise the OFIQ checks show as `skipped`. `evaluation.ipynb` shows how every check does on the labelled London Set.

In [ ]:
import contextlib
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from passphoto_validation.core import PassportPhotoValidator
from passphoto_validation.core.rules import GROUPS
from passphoto_validation.core.utils import prepare_image

IMAGES = Path.cwd().parent / 'tests' / 'test-images'
# OFIQ cloned next to this repo, unless OFIQ_CONFIG_DIR says otherwise
OFIQ_DATA = Path.cwd().parents[1] / 'OFIQ-Project' / 'data'
if 'OFIQ_CONFIG_DIR' not in os.environ and OFIQ_DATA.is_dir():
    os.environ['OFIQ_CONFIG_DIR'] = str(OFIQ_DATA)
STATUS_COLORS = {
    'pass': '#c8e6c9',
    'warn': '#ffe0b2',
    'fail': '#ffcdd2',
    'skipped': '#e0e0e0',
    'error': '#e1bee7',
    'not_checked': '#fafafa',
}

@contextlib.contextmanager
def quiet_native_logs():
    """Hide MediaPipe's C++ log lines, which go straight to the process's stderr."""
    saved = os.dup(2)
    with open(os.devnull, 'w') as devnull:
        os.dup2(devnull.fileno(), 2)
        try:
            yield
        finally:
            os.dup2(saved, 2)
            os.close(saved)


with quiet_native_logs():
    validator = PassportPhotoValidator()
print('OFIQ:', 'loaded' if validator.ofiq else validator.ofiq_unavailable)


def check(image):
    """Run the full validator on a BGR image and return its report."""
    with quiet_native_logs():
        return validator(image)


def load(path):
    """Load a photo the same way the API does (EXIF rotation, colour mode)."""
    return prepare_image(Image.open(path))


def color_status(value):
    return f'background-color: {STATUS_COLORS[value]}' if value in STATUS_COLORS else ''

## Run the photos

`PHOTOS` is every photo in `tests/test-images`; replace it with your own paths to try others. Photos of 2000 px and more take several seconds each.

In [ ]:
PHOTOS = sorted(p for p in IMAGES.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

reports = {path.name: check(load(path)) for path in PHOTOS}
print(f'Checked {len(reports)} photos')

## Overview

One row per photo: the verdict and how many rules got each status. The sample photos are portraits, not ID photos, so most of them rightly fail on head size.

In [ ]:
RULE_STATUSES = ['fail', 'warn', 'pass', 'skipped', 'not_checked']

summary = pd.DataFrame([
    {
        'photo': name,
        'valid': report.is_valid,
        'score': report.overall_score,
        **{status: sum(rule.status == status for rule in report.rules) for status in RULE_STATUSES},
    }
    for name, report in reports.items()
]).set_index('photo')
summary

## Rules × photos

Every rule of the template against every photo. A row that is red or orange for nearly every photo points at a miscalibrated check rather than at bad photos.

In [ ]:
rule_grid = pd.DataFrame({
    name: {(GROUPS[rule.group], rule.title): rule.status for rule in report.rules}
    for name, report in reports.items()
})
rule_grid.index.names = ['group', 'rule']
rule_grid.style.map(color_status)

## Checks × photos

The same one level down, per check, including `error` for a crashed check.

In [ ]:
check_grid = pd.DataFrame({
    name: {result.name: result.status for result in report.checks}
    for name, report in reports.items()
})
check_grid.style.map(color_status)

## One photo in detail

The green box is the detected face. The blue lines are the crown and the chin, which the head-height check measures between.

In [ ]:
PHOTO = '9.jpg'

report = reports[PHOTO]
image = load(IMAGES / PHOTO)
m = report.measurements

fig, ax = plt.subplots(figsize=(5, 6))
ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
if report.bbox:
    b = report.bbox
    ax.add_patch(plt.Rectangle((b['x'], b['y']), b['width'], b['height'], fill=False, color='lime', linewidth=2))
for key in ('crown_y', 'chin_y'):
    if key in m:
        ax.axhline(m[key], color='deepskyblue', linewidth=1.5)
if 'head_height_ratio' in m:
    ax.set_xlabel(f"head height {m['head_height_ratio']:.0%} of the photo (crown: {m['crown_method']})")
ax.set_title(f"{PHOTO}: {'valid' if report.is_valid else 'not valid'}, score {report.overall_score:.0f}")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
rules = pd.DataFrame([
    {
        'group': GROUPS[rule.group],
        'rule': rule.title,
        'status': rule.status,
        'coverage': rule.coverage,
        'reasons': '; '.join(rule.reasons),
        'note': rule.note or '',
    }
    for rule in report.rules
]).set_index(['group', 'rule'])
rules.style.map(color_status, subset=['status'])

In [ ]:
def fmt(value):
    return f'{value:.3g}' if isinstance(value, float) else str(value)


checks = pd.DataFrame([
    {
        'check': result.name,
        'status': result.status,
        'reasons': '; '.join(result.reasons),
        'measurements': ', '.join(f'{key}={fmt(value)}' for key, value in result.measurements.items()),
    }
    for result in report.checks
]).set_index('check')
checks.style.map(color_status, subset=['status'])

## Sanity checks on altered copies

Takes a photo that passes every check and breaks one thing at a time. `expected` is what must happen; `as expected` says whether it did. `also changed` lists every other check that moved: some of that follows from the change (a darkened photo also has low contrast), the rest is noise worth a look.

The old solvePnP head pose moved from about 8° to 14° when the photo was mirrored, and BRISQUE reacted to large flat areas such as the padding around the shrunk head. Both are gone: OFIQ's pose stays within 2° under mirroring, and sharpness is now measured on the face only.

In [ ]:
base = cv2.resize(load(IMAGES / '9.jpg'), (1000, 1000))  # 9.jpg is 2000x2000; smaller is faster
h, w = base.shape[:2]


def rotate(img, degrees):
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), degrees, 1.0)
    return cv2.warpAffine(img, matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)


def shrink(img, factor):
    # Same canvas, smaller person, so the head no longer fills 70-80% of the height
    small = cv2.resize(img, (int(w * factor), int(h * factor)))
    canvas = np.full_like(img, img[5, 5])
    y, x = (h - small.shape[0]) // 2, (w - small.shape[1]) // 2
    canvas[y:y + small.shape[0], x:x + small.shape[1]] = small
    return canvas


# variant: (image, {check: statuses it must end up in})
variants = {
    'mirrored': (cv2.flip(base, 1), {}),
    'darkened (x0.3)': ((base * 0.3).astype(np.uint8), {'exposure': {'fail'}}),
    'brightened (x1.6 +60)': (np.clip(base.astype(int) * 1.6 + 60, 0, 255).astype(np.uint8), {'ofiq_overexposure': {'fail'}}),
    'tilted 20°': (rotate(base, 20), {'ofiq_head_roll': {'fail'}}),
    'blurred': (cv2.GaussianBlur(base, (0, 0), 6), {'face_sharpness': {'fail'}}),
    'head too small': (shrink(base, 0.6), {'face_size': {'fail'}}),
    'no face (flat grey)': (np.full_like(base, 140), {'face_detected': {'fail'}}),
}

baseline = {result.name: result.status for result in check(base).checks}
rows = []
for label, (img, expected) in variants.items():
    statuses = {result.name: result.status for result in check(img).checks}
    gate_failed = statuses['face_detected'] == 'fail'
    also_changed = [
        f'{name} {baseline[name]}→{status}'
        for name, status in statuses.items()
        # Without a face every other check is skipped, which is expected
        if status != baseline[name] and name not in expected and not (gate_failed and status == 'skipped')
    ]
    rows.append({
        'variant': label,
        'expected': ', '.join(f"{name} {'/'.join(sorted(allowed))}" for name, allowed in expected.items()) or 'no change',
        'got': ', '.join(f'{name} {statuses[name]}' for name in expected) or '-',
        'as expected': all(statuses[name] in allowed for name, allowed in expected.items()),
        'also changed': ', '.join(also_changed) or '-',
    })

sanity = pd.DataFrame(rows).set_index('variant')
sanity.style.map(lambda ok: f"background-color: {STATUS_COLORS['pass' if ok else 'fail']}", subset=['as expected'])